# Extracting the IMDB Regression Dataset

This notebook demonstrates how the **Continuous Rating Dataset** was extracted from the raw `aclImdb` folder. 
Unlike the binary classification task (where we just use Positive/Negative), the raw IMDB dataset actually encodes the exact rating (from 1 to 10) in the filename!

The format of each text file in the raw dataset is: `[id]_[rating].txt`

We will extract the `rating` from the filename and pair it with the review text.


In [ ]:
import os
import glob
import pandas as pd
from tqdm.notebook import tqdm

# Define paths to the raw data
# Note: Ensure you have downloaded and extracted the aclImdb dataset into a folder named 'aclImdb'
base_path = "../aclImdb"

def extract_regression_data(split):
    """
    Iterates through both 'pos' and 'neg' folders for a given split (train/test),
    extracts the text and the 1-10 rating from the filename.
    """
    data = []
    
    for sentiment in ['pos', 'neg']:
        folder_path = os.path.join(base_path, split, sentiment)
        if not os.path.exists(folder_path):
            print(f"Directory not found: {folder_path}")
            continue
            
        file_paths = glob.glob(os.path.join(folder_path, "*.txt"))
        
        for file_path in tqdm(file_paths, desc=f"Processing {split}/{sentiment}"):
            # Filename format: id_rating.txt (e.g., 1234_7.txt)
            filename = os.path.basename(file_path)
            rating_str = filename.split('_')[1].replace('.txt', '')
            rating = int(rating_str)
            
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
                
            data.append({"text": text, "rating": rating})
            
    return pd.DataFrame(data)


In [ ]:
# Extract Train and Test data
print("Extracting Training Data...")
train_df = extract_regression_data("train")

print("Extracting Testing Data...")
test_df = extract_regression_data("test")

print(f"\nTrain dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")


In [ ]:
# Preview the extracted regression dataset
train_df.head()


In [ ]:
# Ensure the output directory exists
output_dir = "../processed_for_regression"
os.makedirs(output_dir, exist_ok=True)

# Save to CSV
train_csv_path = os.path.join(output_dir, "imdb_regression_train.csv")
test_csv_path = os.path.join(output_dir, "imdb_regression_test.csv")

train_df.to_csv(train_csv_path, index=False)
test_df.to_csv(test_csv_path, index=False)

print(f"Saved regression data to {output_dir}")
